# LeetCode 786 — K-th Smallest Prime Fraction bằng Python

Link đề: https://leetcode.com/problems/k-th-smallest-prime-fraction/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Binary Search, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def kthSmallestPrimeFraction(self, arr: List[int], k: int) -> List[int]:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một cặp `[tử, mẫu]`; vì các số nguyên tố khác nhau nên mọi phân số đều khác nhau, không có hòa). Notebook dùng `fractions.Fraction` làm **oracle chính xác** để đối chiếu mọi cách giải, kể cả cách nhị phân dùng số thực.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng tăng chặt gồm `1` và các số nguyên tố; xét mọi phân số `arr[i] / arr[j]` với `i < j`; trả về phân số **nhỏ thứ `k`** dưới dạng cặp `[arr[i], arr[j]]`.
2. Nắm các công cụ Python: `fractions.Fraction`, `sorted` với `key`, `heapq` (`heappush`, `heappop`, `heapify`), so sánh phân số bằng **nhân chéo** (không chia), `bisect`, hai con trỏ.
3. Đi từ **liệt kê mọi cặp rồi sắp xếp** → **gộp `n − 1` dãy đã sắp bằng min-heap** → **nhị phân theo giá trị + đếm bằng hai con trỏ** (lời giải cho phần *Follow up*: tốt hơn `O(n²)`) → **nhị phân chính xác bằng `Fraction`**.
4. Hiểu mấu chốt: với mẫu `arr[j]` cố định, `arr[i] / arr[j]` **tăng theo `i`**; với tử cố định thì **giảm theo `j`**. Nhờ tính đơn điệu hai chiều, ta không cần xem hết `n²/2` phân số.
5. Tránh các lỗi kinh điển: nhầm hướng đơn điệu (coi **hàng** là dãy tăng), nhầm chỉ số khi đẩy phần tử kế tiếp vào heap, lệch `k − 1` lần pop, dùng `//` thay `/`, so sánh `<` thay `<=` khi đếm trong nhị phân, và quên rằng đáp án trả về là **cặp số**, không phải giá trị thực.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` **đã sắp xếp tăng chặt**, gồm số `1` và các **số nguyên tố**, mọi phần tử **đôi một khác nhau**; cho số nguyên `k`.

Với mọi cặp chỉ số `0 <= i < j < arr.length`, xét phân số `arr[i] / arr[j]`. Hãy trả về phân số **nhỏ thứ `k`** (đếm từ 1) dưới dạng mảng hai số nguyên `[arr[i], arr[j]]`.

**Ví dụ 1:** `arr = [1,2,3,5]`, `k = 3` → `[2,5]`. Các phân số xếp tăng: `1/5, 1/3, 2/5, 1/2, 3/5, 2/3`; phân số thứ ba là `2/5`.

**Ví dụ 2:** `arr = [1,7]`, `k = 1` → `[1,7]`.

**Ràng buộc:**

- `2 <= arr.length <= 1000`
- `1 <= arr[i] <= 3 * 10^4`, `arr[0] == 1`, `arr[i]` nguyên tố với `i > 0`, tăng chặt.
- `1 <= k <= arr.length * (arr.length - 1) / 2`

**Follow up:** có thể giải tốt hơn `O(n²)` không?

In [1]:
import heapq
import random
import time
from bisect import bisect_left
from fractions import Fraction
from itertools import combinations
from typing import List

ex1 = ([1, 2, 3, 5], 3)
ex2 = ([1, 7], 1)

arr, k = ex1
fracs = sorted((Fraction(arr[i], arr[j]), arr[i], arr[j]) for i in range(len(arr)) for j in range(i + 1, len(arr)))
print("các phân số xếp tăng:", [f"{a}/{b}" for _, a, b in fracs])
print("phân số thứ", k, ":", fracs[k - 1][1:], "| tổng số phân số =", len(arr) * (len(arr) - 1) // 2)

các phân số xếp tăng: ['1/5', '1/3', '2/5', '1/2', '3/5', '2/3']
phân số thứ 3 : (2, 5) | tổng số phân số = 6


## 3. Kiến thức nền

### 3.1 Phân số trong Python: `Fraction` và so sánh nhân chéo

Số thực dấu phẩy động có sai số (`1/3` không biểu diễn chính xác). `fractions.Fraction(a, b)` giữ phân số **đúng**, tự rút gọn, và so sánh được bằng `<`, `==`. Nếu không dùng `Fraction`, ta so sánh `a/b < c/d` bằng **nhân chéo**: vì `b, d > 0`, `a/b < c/d ⇔ a·d < c·b` (chỉ cần số nguyên).

In [2]:
print(Fraction(2, 6), Fraction(2, 5) < Fraction(1, 2), 0.1 + 0.2 == 0.3, Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10))

a, b, c, d = 2, 5, 1, 2          # so sánh 2/5 với 1/2
print("2/5 < 1/2 ?", a * d < c * b, "(nhân chéo:", a * d, "<", c * b, ")")

1/3 True False True
2/5 < 1/2 ? True (nhân chéo: 4 < 5 )


### 3.2 Sắp xếp theo khoá (`key`) và `sorted` với tuple

`sorted(items, key=f)` sắp theo giá trị `f(item)`. Tuple so sánh **từng phần tử từ trái sang phải**: `(0.4, 2, 5) < (0.5, 1, 2)`. Ta thường đóng gói `(giá trị phân số, tử, mẫu)` để sắp rồi đọc ra cặp `(tử, mẫu)`.

In [3]:
pairs = [(2, 5), (1, 2), (3, 5), (1, 5)]
print(sorted(pairs, key=lambda p: p[0] / p[1]))
print(sorted((p[0] / p[1], p[0], p[1]) for p in pairs)[:2])

[(1, 5), (2, 5), (1, 2), (3, 5)]
[(0.2, 1, 5), (0.4, 2, 5)]


### 3.3 Min-heap `heapq`

`heapq` biến một `list` thành **min-heap**: phần tử **nhỏ nhất** luôn ở `heap[0]`. `heappush(h, x)` thêm `x` (`O(log n)`), `heappop(h)` lấy và bỏ phần tử nhỏ nhất (`O(log n)`). Khi cần "lấy lần lượt các phần tử nhỏ nhất từ nhiều dãy đã sắp", heap là công cụ chuẩn (**gộp `n` dãy**).

In [4]:
h = []
for x in [5, 1, 4, 2, 3]:
    heapq.heappush(h, x)
print("đỉnh heap:", h[0], "| lấy lần lượt:", [heapq.heappop(h) for _ in range(5)])

# gộp 3 dãy đã sắp bằng heap: mỗi dãy chỉ đưa phần tử đầu vào heap, lấy ra rồi đưa phần tử kế tiếp của dãy đó
chains = [[1, 4, 9], [2, 3, 10], [5, 6, 7]]
heap = [(c[0], idx, 0) for idx, c in enumerate(chains)]
heapq.heapify(heap)
merged = []
while heap:
    v, idx, pos = heapq.heappop(heap)
    merged.append(v)
    if pos + 1 < len(chains[idx]):
        heapq.heappush(heap, (chains[idx][pos + 1], idx, pos + 1))
print("gộp:", merged)

đỉnh heap: 1 | lấy lần lượt: [1, 2, 3, 4, 5]
gộp: [1, 2, 3, 4, 5, 6, 7, 9, 10]


### 3.4 Nhị phân theo **giá trị** (không phải theo chỉ số)

Ta quen nhị phân trên chỉ số mảng. Ở đây ta nhị phân trên **trục giá trị** `x ∈ [0, 1]`: hàm `count(x)` = "số phân số `≤ x`" **không giảm** theo `x`. Bài toán "phân số nhỏ thứ `k`" trở thành: tìm `x` nhỏ nhất sao cho `count(x) >= k`; khi `count(x) == k` thì phân số lớn nhất `≤ x` chính là đáp án. Đây là kỹ thuật "nhị phân trên đáp án" (*binary search on answer*), dùng khi đếm "có bao nhiêu phần tử ≤ x" nhanh hơn liệt kê.

### 3.5 Hai con trỏ

Với mẫu `arr[j]` tăng dần theo `j`, ngưỡng `x · arr[j]` cũng tăng; số tử `arr[i]` thoả `arr[i] <= x·arr[j]` chỉ **tăng** theo `j`. Vì vậy con trỏ `i` chỉ **đi tiến**, không bao giờ lùi: tổng thao tác `O(n)` cho mỗi lần đếm.

In [5]:
arr = [1, 2, 3, 5, 7, 11]
x = 0.5
i = 0
for j in range(1, len(arr)):
    while i < j and arr[i] <= x * arr[j]:
        i += 1
    print(f"mẫu {arr[j]:2d}: ngưỡng x*mẫu = {x * arr[j]:4.1f} -> có {i} tử thoả (tử {arr[:i]})")

mẫu  2: ngưỡng x*mẫu =  1.0 -> có 1 tử thoả (tử [1])
mẫu  3: ngưỡng x*mẫu =  1.5 -> có 1 tử thoả (tử [1])
mẫu  5: ngưỡng x*mẫu =  2.5 -> có 2 tử thoả (tử [1, 2])
mẫu  7: ngưỡng x*mẫu =  3.5 -> có 3 tử thoả (tử [1, 2, 3])
mẫu 11: ngưỡng x*mẫu =  5.5 -> có 4 tử thoả (tử [1, 2, 3, 5])


## 4. Ý tưởng nền tảng / trực giác

Bảng các phân số `arr[i] / arr[j]` cho `arr = [1,2,3,5]` (hàng là tử, cột là mẫu, chỉ lấy `i < j`):

| tử \ mẫu | 2 | 3 | 5 |
|---|---|---|---|
| 1 | 1/2 | 1/3 | 1/5 |
| 2 | — | 2/3 | 2/5 |
| 3 | — | — | 3/5 |

Quan sát bằng mắt:

- **Trong một cột** (mẫu cố định), đi xuống thì tử tăng nên phân số **tăng**: `1/5 < 2/5 < 3/5`.
- **Trong một hàng** (tử cố định), đi sang phải thì mẫu tăng nên phân số **giảm**: `1/2 > 1/3 > 1/5`.

Vậy bảng "gần như sắp xếp" theo hai chiều, như bài *k-th smallest in sorted matrix*. Mỗi cột là một dãy **đã sắp xếp tăng**; bài toán là **gộp các cột** và lấy phần tử thứ `k`.

Thử tay với `k = 3`: ba phân số nhỏ nhất là `1/5, 1/3, 2/5` → đáp án `2/5`.

In [6]:
arr = [1, 2, 3, 5]
n = len(arr)
print("      " + "".join(f"{arr[j]:>6}" for j in range(n)))
for i in range(n):
    print(f"{arr[i]:>4}  " + "".join((f"{arr[i]}/{arr[j]}".rjust(6) if i < j else "     .") for j in range(n)))
print("cột j cố định: tăng khi i tăng ->", [round(arr[i] / arr[3], 3) for i in range(3)])
print("hàng i cố định: giảm khi j tăng ->", [round(arr[0] / arr[j], 3) for j in range(1, 4)])

           1     2     3     5
   1       .   1/2   1/3   1/5
   2       .     .   2/3   2/5
   3       .     .     .   3/5
   5       .     .     .     .
cột j cố định: tăng khi i tăng -> [0.2, 0.4, 0.6]
hàng i cố định: giảm khi j tăng -> [0.5, 0.333, 0.2]


## 5. Cách 1 — Brute Force: liệt kê mọi phân số rồi sắp xếp

**Ý tưởng:** sinh `n(n−1)/2` phân số, sắp tăng, lấy phần tử thứ `k`. Để so sánh **chính xác** ta dùng `Fraction` làm khoá (hoặc nhân chéo).

In [7]:
def kth_brute(arr, k):
    items = []
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            items.append((Fraction(arr[i], arr[j]), arr[i], arr[j]))
    items.sort()
    return [items[k - 1][1], items[k - 1][2]]


print(kth_brute(*ex1), kth_brute(*ex2))
assert kth_brute(*ex1) == [2, 5] and kth_brute(*ex2) == [1, 7]

[2, 5] [1, 7]


**Độ phức tạp:** thời gian `O(n² log n)`, bộ nhớ `O(n²)` (với `n = 1000` là ~500.000 phân số, ổn nhưng lãng phí). Ta dùng nó làm **oracle** vì rõ ràng đúng theo đề.

**Nút thắt:** ta lưu và sắp **toàn bộ** phân số trong khi chỉ cần phần tử thứ `k`. Nếu `k` nhỏ ta chỉ cần xem những phân số rất bé. Tính đơn điệu theo cột cho phép "gộp" thay vì "sắp lại tất cả".

## 6. Ý tưởng thuật toán / bước đột phá

Có hai hướng chính:

1. **Heap gộp `n − 1` cột đã sắp:** cột `j` là dãy `arr[0]/arr[j] < arr[1]/arr[j] < ...< arr[j−1]/arr[j]` (tăng). Đưa phần tử đầu mỗi cột vào min-heap, lấy ra phần tử nhỏ nhất `k − 1` lần (mỗi lần lấy xong đưa phần tử kế tiếp **cùng cột** vào heap); phần tử ở đỉnh heap sau đó là đáp án. Thời gian `O((n + k) log n)`.
2. **Nhị phân trên giá trị `x`:** `count(x)` (số phân số `≤ x`) đếm được trong `O(n)` bằng hai con trỏ nhờ tính đơn điệu. Nhị phân `x` trong `[0, 1]` tới khi `count(x) == k` thì đáp án là phân số lớn nhất `≤ x`. Thời gian `O(n log(1/ε))` — **tốt hơn `O(n²)`** (trả lời Follow up).

Hai ý tưởng này đều né việc liệt kê toàn bộ `n²/2` phân số.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm: **Cách 2** min-heap gộp các cột; **Cách 3** nhị phân trên giá trị bằng số thực + hai con trỏ; **Cách 4** nhị phân với `Fraction` (chính xác tuyệt đối).

### Cách 2 — Min-heap gộp `n − 1` cột đã sắp

**Giải thích từng đoạn:**

1. Với mỗi cột `j = 1..n−1`, đưa `(arr[0] / arr[j], 0, j)` vào heap (phần tử nhỏ nhất của cột `j`).
2. Lặp `k − 1` lần: lấy phần tử nhỏ nhất `(giá trị, i, j)`; nếu `i + 1 < j` đưa `(arr[i+1] / arr[j], i+1, j)` vào heap.
3. Phần tử ở đỉnh heap bây giờ là phân số nhỏ thứ `k`; trả về `[arr[i], arr[j]]`.

Vì các số khác nhau và là nguyên tố (hoặc `1`) nên mọi phân số **đôi một khác nhau**; ta có thể dùng số thực `arr[i]/arr[j]` làm khoá (hai phân số khác nhau cách nhau ít nhất `1/(3·10^4)² ≈ 1,1·10^-9`, lớn hơn rất xa sai số `10^-16` của số thực kép). Để **chắc chắn tuyệt đối** ta cũng viết bản dùng nhân chéo bằng lớp bao.

In [8]:
def kth_heap(arr, k):
    n = len(arr)
    heap = [(arr[0] / arr[j], 0, j) for j in range(1, n)]
    heapq.heapify(heap)
    for _ in range(k - 1):
        _, i, j = heapq.heappop(heap)
        if i + 1 < j:
            heapq.heappush(heap, (arr[i + 1] / arr[j], i + 1, j))
    _, i, j = heap[0]
    return [arr[i], arr[j]]


class Frac:
    """Phân số với so sánh nhân chéo (chính xác, không dùng số thực)."""
    __slots__ = ("num", "den", "i", "j")

    def __init__(self, num, den, i, j):
        self.num, self.den, self.i, self.j = num, den, i, j

    def __lt__(self, other):
        return self.num * other.den < other.num * self.den


def kth_heap_exact(arr, k):
    n = len(arr)
    heap = [Frac(arr[0], arr[j], 0, j) for j in range(1, n)]
    heapq.heapify(heap)
    for _ in range(k - 1):
        top = heapq.heappop(heap)
        if top.i + 1 < top.j:
            heapq.heappush(heap, Frac(arr[top.i + 1], arr[top.j], top.i + 1, top.j))
    top = heap[0]
    return [arr[top.i], arr[top.j]]


print(kth_heap(*ex1), kth_heap(*ex2), kth_heap_exact(*ex1), kth_heap_exact(*ex2))
assert kth_heap(*ex1) == kth_heap_exact(*ex1) == [2, 5] and kth_heap(*ex2) == kth_heap_exact(*ex2) == [1, 7]

[2, 5] [1, 7] [2, 5] [1, 7]


In [9]:
# Code nộp LeetCode cho Cách 2
import heapq


class SolutionHeap:
    def kthSmallestPrimeFraction(self, arr: List[int], k: int) -> List[int]:
        n = len(arr)
        heap = [(arr[0] / arr[j], 0, j) for j in range(1, n)]
        heapq.heapify(heap)
        for _ in range(k - 1):
            _, i, j = heapq.heappop(heap)
            if i + 1 < j:
                heapq.heappush(heap, (arr[i + 1] / arr[j], i + 1, j))
        _, i, j = heap[0]
        return [arr[i], arr[j]]

**Độ phức tạp:** khởi tạo heap `O(n)`; mỗi trong `k − 1` lần: `pop` + `push` `O(log n)`. Tổng `O(n + k log n)`; bộ nhớ `O(n)`. Khi `k` rất lớn (cỡ `n²/2`) thì `O(n² log n)`, không tốt hơn brute — đó là lý do cần Cách 3.

### Cách 3 — Nhị phân trên giá trị + đếm bằng hai con trỏ

**Giải thích từng đoạn:**

1. Khoảng tìm `[lo, hi] = [0, 1]`. Lấy `mid = (lo + hi) / 2`.
2. **Đếm** số phân số `≤ mid`: với `j` từ `1` đến `n−1`, đẩy con trỏ `i` (chỉ tiến) tới khi `arr[i] > mid · arr[j]` hoặc `i == j`; khi đó có đúng `i` tử thoả (`arr[0..i−1]`), cộng `i` vào `count`. Đồng thời ghi nhận phân số **lớn nhất** `≤ mid` là `arr[i−1] / arr[j]` (so sánh bằng nhân chéo).
3. Nếu `count == k`: trả `[tử, mẫu]` của phân số lớn nhất ghi nhận. Nếu `count < k`: cần giá trị lớn hơn → `lo = mid`. Nếu `count > k`: `hi = mid`.

Vì chỉ có hữu hạn phân số và hai phân số khác nhau cách nhau `≥ 1,1·10^-9`, nhị phân sẽ rơi đúng vào khoảng mà `count == k` sau khoảng 40 vòng.

In [10]:
def kth_bsearch(arr, k, return_rounds=False):
    n = len(arr)
    lo, hi = 0.0, 1.0
    rounds = 0
    while True:
        rounds += 1
        mid = (lo + hi) / 2
        count = 0
        i = 0
        best_num, best_den = 0, 1                 # phân số lớn nhất <= mid, khởi tạo 0/1
        for j in range(1, n):
            while i < j and arr[i] <= mid * arr[j]:
                i += 1
            count += i
            if i > 0 and arr[i - 1] * best_den > best_num * arr[j]:     # arr[i-1]/arr[j] > best?
                best_num, best_den = arr[i - 1], arr[j]
        if count == k:
            res = [best_num, best_den]
            return (res, rounds) if return_rounds else res
        if count < k:
            lo = mid
        else:
            hi = mid


print(kth_bsearch(*ex1), kth_bsearch(*ex2))
res, rounds = kth_bsearch(*ex1, return_rounds=True)
print("số vòng nhị phân ở ví dụ 1:", rounds)
assert kth_bsearch(*ex1) == [2, 5] and kth_bsearch(*ex2) == [1, 7]

[2, 5] [1, 7]
số vòng nhị phân ở ví dụ 1: 4


In [11]:
# Code nộp LeetCode cho Cách 3
class SolutionBinary:
    def kthSmallestPrimeFraction(self, arr: List[int], k: int) -> List[int]:
        n = len(arr)
        lo, hi = 0.0, 1.0
        while True:
            mid = (lo + hi) / 2
            count = 0
            i = 0
            bn, bd = 0, 1
            for j in range(1, n):
                while i < j and arr[i] <= mid * arr[j]:
                    i += 1
                count += i
                if i > 0 and arr[i - 1] * bd > bn * arr[j]:
                    bn, bd = arr[i - 1], arr[j]
            if count == k:
                return [bn, bd]
            if count < k:
                lo = mid
            else:
                hi = mid

**Độ phức tạp:** mỗi lần đếm `O(n)` (hai con trỏ); số vòng nhị phân `O(log(1/δ))` với `δ ≈ 10^-9` nên khoảng 30–40 vòng. Tổng `O(n log(1/δ))` — **không phụ thuộc `k`**, trả lời Follow up. Bộ nhớ `O(1)`.

### Cách 4 — Nhị phân với `Fraction` (chính xác tuyệt đối)

Cách 3 dựa vào lập luận "số thực đủ chính xác". Nếu không muốn tin vào lập luận, thay `mid` bằng một `Fraction` (cận `lo, hi` cũng là `Fraction`) và so sánh bằng **nhân chéo số nguyên**: `arr[i] <= mid · arr[j] ⇔ arr[i] · mid.denominator <= mid.numerator · arr[j]`. Mọi phép tính đều đúng tuyệt đối; mẫu số của `mid` tăng gấp đôi mỗi vòng (`2^40` sau 40 vòng — số nguyên Python vẫn nhỏ).

In [12]:
def kth_bsearch_exact(arr, k):
    n = len(arr)
    lo, hi = Fraction(0), Fraction(1)
    while True:
        mid = (lo + hi) / 2
        p, q = mid.numerator, mid.denominator     # mid = p / q
        count = 0
        i = 0
        best_num, best_den = 0, 1
        for j in range(1, n):
            while i < j and arr[i] * q <= p * arr[j]:        # arr[i]/arr[j] <= p/q
                i += 1
            count += i
            if i > 0 and arr[i - 1] * best_den > best_num * arr[j]:
                best_num, best_den = arr[i - 1], arr[j]
        if count == k:
            return [best_num, best_den]
        if count < k:
            lo = mid
        else:
            hi = mid


print(kth_bsearch_exact(*ex1), kth_bsearch_exact(*ex2))
assert kth_bsearch_exact(*ex1) == [2, 5] and kth_bsearch_exact(*ex2) == [1, 7]

[2, 5] [1, 7]


In [13]:
# Code nộp LeetCode cho Cách 4
from fractions import Fraction


class SolutionExact:
    def kthSmallestPrimeFraction(self, arr: List[int], k: int) -> List[int]:
        n = len(arr)
        lo, hi = Fraction(0), Fraction(1)
        while True:
            mid = (lo + hi) / 2
            p, q = mid.numerator, mid.denominator
            count, i = 0, 0
            bn, bd = 0, 1
            for j in range(1, n):
                while i < j and arr[i] * q <= p * arr[j]:
                    i += 1
                count += i
                if i > 0 and arr[i - 1] * bd > bn * arr[j]:
                    bn, bd = arr[i - 1], arr[j]
            if count == k:
                return [bn, bd]
            if count < k:
                lo = mid
            else:
                hi = mid

**Độ phức tạp:** cùng bậc `O(n log(1/δ))`, hằng số lớn hơn một chút vì số nguyên lớn. Dùng khi bạn cần **bảo đảm** đúng mà không dựa vào độ chính xác của số thực.

## 8. TRACE từng bước

**Trace Cách 2 (heap)** với `arr = [1,2,3,5]`, `k = 3`. Ban đầu heap chứa phần tử đầu mỗi cột `j = 1, 2, 3`.

| bước | heap trước (giá trị, `i`, `j`) | lấy ra | đưa vào | heap sau |
|---|---|---|---|---|
| khởi tạo | — | — | `1/2 (0,1)`, `1/3 (0,2)`, `1/5 (0,3)` | `{1/5, 1/3, 1/2}` |
| 1 (lần pop 1 trong `k−1=2`) | `{1/5, 1/3, 1/2}` | `1/5 (i=0,j=3)` | `2/5 (1,3)` vì `i+1=1 < 3` | `{1/3, 2/5, 1/2}` |
| 2 | `{1/3, 2/5, 1/2}` | `1/3 (0,2)` | `2/3 (1,2)` vì `1 < 2` | `{2/5, 1/2, 2/3}` |
| kết thúc | đỉnh heap = `2/5` | | | trả `[2,5]` |

Hàm trace in từng bước:

In [14]:
def trace_heap(arr, k):
    n = len(arr)
    heap = [(arr[0] / arr[j], 0, j) for j in range(1, n)]
    heapq.heapify(heap)
    show = lambda h: sorted((f"{arr[i]}/{arr[j]}" for _, i, j in h), key=lambda s: eval(s))
    print("arr =", arr, "k =", k)
    print("khởi tạo heap:", show(heap))
    for step in range(1, k):
        v, i, j = heapq.heappop(heap)
        msg = f"bước {step}: lấy {arr[i]}/{arr[j]} (là phân số nhỏ thứ {step})"
        if i + 1 < j:
            heapq.heappush(heap, (arr[i + 1] / arr[j], i + 1, j))
            msg += f", đưa {arr[i + 1]}/{arr[j]} vào"
        else:
            msg += ", cột đã hết"
        print(msg, "| heap:", show(heap))
    v, i, j = heap[0]
    print(f"đỉnh heap = {arr[i]}/{arr[j]} là phân số nhỏ thứ {k}")
    return [arr[i], arr[j]]


assert trace_heap([1, 2, 3, 5], 3) == [2, 5]
print()
assert trace_heap([1, 2, 3, 5, 7], 7) == kth_brute([1, 2, 3, 5, 7], 7)

arr = [1, 2, 3, 5] k = 3
khởi tạo heap: ['1/5', '1/3', '1/2']
bước 1: lấy 1/5 (là phân số nhỏ thứ 1), đưa 2/5 vào | heap: ['1/3', '2/5', '1/2']
bước 2: lấy 1/3 (là phân số nhỏ thứ 2), đưa 2/3 vào | heap: ['2/5', '1/2', '2/3']
đỉnh heap = 2/5 là phân số nhỏ thứ 3

arr = [1, 2, 3, 5, 7] k = 7
khởi tạo heap: ['1/7', '1/5', '1/3', '1/2']
bước 1: lấy 1/7 (là phân số nhỏ thứ 1), đưa 2/7 vào | heap: ['1/5', '2/7', '1/3', '1/2']
bước 2: lấy 1/5 (là phân số nhỏ thứ 2), đưa 2/5 vào | heap: ['2/7', '1/3', '2/5', '1/2']
bước 3: lấy 2/7 (là phân số nhỏ thứ 3), đưa 3/7 vào | heap: ['1/3', '2/5', '3/7', '1/2']
bước 4: lấy 1/3 (là phân số nhỏ thứ 4), đưa 2/3 vào | heap: ['2/5', '3/7', '1/2', '2/3']
bước 5: lấy 2/5 (là phân số nhỏ thứ 5), đưa 3/5 vào | heap: ['3/7', '1/2', '3/5', '2/3']
bước 6: lấy 3/7 (là phân số nhỏ thứ 6), đưa 5/7 vào | heap: ['1/2', '3/5', '2/3', '5/7']
đỉnh heap = 1/2 là phân số nhỏ thứ 7


**Trace Cách 3 (nhị phân)** với `arr = [1,2,3,5]`, `k = 3`:

| vòng | `lo` | `hi` | `mid` | `count(≤ mid)` | hành động |
|---|---|---|---|---|---|
| 1 | 0 | 1 | 0,5 | 4 (`1/5, 1/3, 2/5, 1/2`) | `count > k = 3` → `hi = 0,5` |
| 2 | 0 | 0,5 | 0,25 | 1 (`1/5`) | `count < k` → `lo = 0,25` |
| 3 | 0,25 | 0,5 | 0,375 | 2 (`1/5, 1/3`) | `count < k` → `lo = 0,375` |
| 4 | 0,375 | 0,5 | 0,4375 | 3 (`1/5, 1/3, 2/5`) | `count == k` → đáp án = phân số lớn nhất `≤ 0,4375`, tức `2/5` |

Hàm dưới in số đếm thật ở từng vòng (số tử thoả với mỗi mẫu `arr[j]`):

In [15]:
def trace_bsearch(arr, k):
    n = len(arr)
    lo, hi = 0.0, 1.0
    vong = 0
    print("arr =", arr, "k =", k)
    while True:
        vong += 1
        mid = (lo + hi) / 2
        count, i = 0, 0
        bn, bd = 0, 1
        per_j = []
        for j in range(1, n):
            while i < j and arr[i] <= mid * arr[j]:
                i += 1
            count += i
            per_j.append(f"j={j}(mẫu {arr[j]}): {i} tử")
            if i > 0 and arr[i - 1] * bd > bn * arr[j]:
                bn, bd = arr[i - 1], arr[j]
        print(f"vòng {vong}: lo={lo:.6f} hi={hi:.6f} mid={mid:.6f} | " + "; ".join(per_j) + f" | count={count}, lớn nhất<=mid = {bn}/{bd}")
        if count == k:
            print("count == k -> đáp án", [bn, bd])
            return [bn, bd]
        if count < k:
            lo = mid
        else:
            hi = mid


assert trace_bsearch([1, 2, 3, 5], 3) == [2, 5]

arr = [1, 2, 3, 5] k = 3
vòng 1: lo=0.000000 hi=1.000000 mid=0.500000 | j=1(mẫu 2): 1 tử; j=2(mẫu 3): 1 tử; j=3(mẫu 5): 2 tử | count=4, lớn nhất<=mid = 1/2
vòng 2: lo=0.000000 hi=0.500000 mid=0.250000 | j=1(mẫu 2): 0 tử; j=2(mẫu 3): 0 tử; j=3(mẫu 5): 1 tử | count=1, lớn nhất<=mid = 1/5
vòng 3: lo=0.250000 hi=0.500000 mid=0.375000 | j=1(mẫu 2): 0 tử; j=2(mẫu 3): 1 tử; j=3(mẫu 5): 1 tử | count=2, lớn nhất<=mid = 1/3
vòng 4: lo=0.375000 hi=0.500000 mid=0.437500 | j=1(mẫu 2): 0 tử; j=2(mẫu 3): 1 tử; j=3(mẫu 5): 2 tử | count=3, lớn nhất<=mid = 2/5
count == k -> đáp án [2, 5]


## 9. Code nộp LeetCode

Chọn **nhị phân trên giá trị + hai con trỏ**: `O(n log(1/δ))`, không phụ thuộc `k`, đáp ứng Follow up. Giữ nguyên `class Solution`, `kthSmallestPrimeFraction(self, arr, k)`:

In [16]:
from typing import List


class Solution:
    def kthSmallestPrimeFraction(self, arr: List[int], k: int) -> List[int]:
        n = len(arr)
        lo, hi = 0.0, 1.0
        while True:
            mid = (lo + hi) / 2                      # đoán giá trị phân số
            count = 0                                # số phân số <= mid
            i = 0
            bn, bd = 0, 1                            # phân số lớn nhất <= mid
            for j in range(1, n):
                while i < j and arr[i] <= mid * arr[j]:
                    i += 1                           # i chỉ tiến: hai con trỏ
                count += i                           # các tử arr[0..i-1] thoả với mẫu arr[j]
                if i > 0 and arr[i - 1] * bd > bn * arr[j]:
                    bn, bd = arr[i - 1], arr[j]
            if count == k:
                return [bn, bd]
            if count < k:
                lo = mid
            else:
                hi = mid


print(Solution().kthSmallestPrimeFraction([1, 2, 3, 5], 3))   # [2, 5]
print(Solution().kthSmallestPrimeFraction([1, 7], 1))         # [1, 7]
assert Solution().kthSmallestPrimeFraction([1, 2, 3, 5], 3) == [2, 5]
assert Solution().kthSmallestPrimeFraction([1, 7], 1) == [1, 7]

[2, 5]
[1, 7]


## 10. Vì sao đúng

**Cách 2 (heap).** Bất biến: sau `t` lần pop, các phần tử đã lấy là đúng `t` phân số nhỏ nhất, và heap chứa, với mỗi cột, phần tử **nhỏ nhất chưa lấy** của cột đó. Thật vậy, phân số nhỏ nhất chưa lấy của toàn bảng phải là phần tử nhỏ nhất chưa lấy của **cột chứa nó** (cột tăng dần), tức nằm trong heap; heap lấy phần tử nhỏ nhất của chúng. Sau khi lấy, ta đẩy phần tử kế tiếp cùng cột để giữ bất biến. Sau `k − 1` lần pop, đỉnh heap là phân số nhỏ thứ `k`.

**Cách 3 (nhị phân).** `count(x) = #{(i, j): arr[i]/arr[j] ≤ x}` không giảm theo `x`. Với `j` cố định, các tử thoả `arr[i] ≤ x·arr[j]` là một tiền tố `arr[0..i−1]` (vì `arr` tăng), nên đếm = chiều dài tiền tố. Chiều dài này không giảm theo `j` (ngưỡng `x·arr[j]` tăng) nên hai con trỏ đúng; ràng buộc `i < j` đảm bảo chỉ đếm cặp `i < j`. Khi `count(mid) = k` thì đúng `k` phân số `≤ mid`, nên phân số **lớn nhất** trong số đó chính là phân số nhỏ thứ `k`. Vì các phân số đôi một khác nhau, hàm `count` nhận mọi giá trị `0..n(n−1)/2` khi `x` chạy từ `0` tới `1`, nên với mọi `k` hợp lệ luôn tồn tại `mid` thoả `count = k`, và nhị phân tìm thấy sau hữu hạn vòng: khoảng `[lo, hi]` luôn thoả `count(lo) < k ≤ count(hi)`, hẹp dần tới khi chứa đúng một phân số.

**Độ chính xác của số thực.** Hai phân số khác nhau `a/b ≠ c/d` với `b, d ≤ 3·10^4` cách nhau `|ad − bc| / (bd) ≥ 1/(9·10^8) ≈ 1,1·10^-9`. Sai số làm tròn tương đối của số thực kép là `~10^-16`, nhỏ hơn khoảng cách này nhiều bậc, nên các phép so sánh `arr[i] <= mid * arr[j]` và khoá `arr[i]/arr[j]` trong heap không bao giờ làm đảo thứ tự hai phân số khác nhau. Ta không chỉ dựa vào lập luận này: ta **kiểm chứng** bằng oracle `Fraction` trên hàng nghìn input ở mục 12, và Cách 4 dùng số học hoàn toàn chính xác.

## 11. Time / Space Complexity

Đặt `n = len(arr)`, `m = n(n−1)/2` là số phân số.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Liệt kê + sắp xếp | `O(n² log n)` | `O(n²)` | chỉ làm oracle |
| 2. Min-heap gộp cột | `O(n + k log n)` | `O(n)` | tốt khi `k` nhỏ |
| 3. Nhị phân + hai con trỏ | `O(n log(1/δ))` | `O(1)` | **đáp án chính**; không phụ thuộc `k` |
| 4. Nhị phân `Fraction` | `O(n log(1/δ))` | `O(1)` | chính xác tuyệt đối, hằng số lớn hơn |

Đo thực tế trên `n = 1000` (mảng `1` và 999 số nguyên tố đầu tiên), với `k` nhỏ, vừa, lớn:

In [17]:
def primes_up_to(limit):
    sieve = bytearray([1]) * (limit + 1)
    sieve[0:2] = b"\x00\x00"
    for p in range(2, int(limit ** 0.5) + 1):
        if sieve[p]:
            sieve[p * p::p] = bytearray(len(sieve[p * p::p]))
    return [p for p in range(limit + 1) if sieve[p]]


PRIMES = primes_up_to(30000)
print("số nguyên tố <= 30000:", len(PRIMES))
big = [1] + PRIMES[:999]
m_total = len(big) * (len(big) - 1) // 2
print("n =", len(big), "| số phân số =", m_total)


def timeit(fn, *args):
    t0 = time.perf_counter()
    res = fn(*args)
    return res, time.perf_counter() - t0


for kk in (10, 1000, m_total // 2, m_total):
    print(f"k = {kk:7d}:", end=" ")
    results = []
    for name, fn in [("heap", kth_heap), ("binary", kth_bsearch), ("exact", kth_bsearch_exact)]:
        res, dt = timeit(fn, big, kk)
        results.append(res)
        print(f"{name} {dt * 1000:8.1f} ms", end=" | ")
    assert results[0] == results[1] == results[2], kk
    print("đáp án", results[0])
res, dt = timeit(kth_brute, big, 1000)
print(f"brute (k=1000) {dt * 1000:.0f} ms ->", res)

số nguyên tố <= 30000: 3245
n = 1000 | số phân số = 499500
k =      10: heap      0.5 ms | binary      4.4 ms | exact      3.9 ms | đáp án [1, 7829]
k =    1000: heap      0.9 ms | binary      5.6 ms | exact      5.1 ms | đáp án [3, 7541]
k =  249750: 

heap    213.5 ms | binary      6.7 ms | exact      5.3 ms | đáp án [1619, 3631]
k =  499500: 

heap    383.6 ms | binary      2.4 ms | exact      2.5 ms | đáp án [7877, 7879]


brute (k=1000) 4005 ms -> [3, 7541]


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) vét cạn mọi mảng nhỏ và mọi `k`, (E) ngẫu nhiên có seed đúng ràng buộc, (F) giới hạn lớn nhất. Oracle: `kth_brute` dùng `Fraction`.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "brute": kth_brute,
    "heap": kth_heap,
    "heap_exact": kth_heap_exact,
    "binary": kth_bsearch,
    "exact": kth_bsearch_exact,
    "Solution": lambda a, k: Solution().kthSmallestPrimeFraction(a, k),
    "SolutionHeap": lambda a, k: SolutionHeap().kthSmallestPrimeFraction(a, k),
    "SolutionBinary": lambda a, k: SolutionBinary().kthSmallestPrimeFraction(a, k),
    "SolutionExact": lambda a, k: SolutionExact().kthSmallestPrimeFraction(a, k),
}


def check(arr, k, want, names=None):
    for name in (names or IMPLS):
        got = IMPLS[name](list(arr), k)
        assert got == want, (name, arr, k, got, want)


check(*ex1, [2, 5])
check(*ex2, [1, 7])
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt đúng")

ví dụ 1, 2: cả 9 cài đặt đúng


### B. Edge case

- `n = 2`: chỉ có một phân số `1/p`.
- `k = 1` (phân số nhỏ nhất luôn là `1/arr[-1]`) và `k = m` (phân số lớn nhất là tỉ số liền kề lớn nhất `max arr[j-1]/arr[j]`, không nhất thiết là cặp cuối).
- Mảng chỉ gồm các số nguyên tố liên tiếp đầu tiên.
- Mảng có số nguyên tố lớn nhất `29989` (gần giới hạn `3·10^4`).
- `k` ở giữa.

In [19]:
check([1, 2], 1, [1, 2])
check([1, 29989], 1, [1, 29989])
for arr in ([1, 2, 3], [1, 2, 3, 5, 7], [1, 3, 7, 11, 13, 17], [1, 2, 29983, 29989]):
    m = len(arr) * (len(arr) - 1) // 2
    check(arr, 1, [1, arr[-1]])                     # nhỏ nhất: 1/arr[-1]
    best = max(range(1, len(arr)), key=lambda j: Fraction(arr[j - 1], arr[j]))
    check(arr, m, [arr[best - 1], arr[best]])        # lớn nhất: tỉ số liền kề lớn nhất
    for kk in range(1, m + 1):
        check(arr, kk, kth_brute(arr, kk))
print("edge case OK: k = 1 luôn là 1/arr[-1], k = m là tỉ số liền kề lớn nhất")

edge case OK: k = 1 luôn là 1/arr[-1], k = m là tỉ số liền kề lớn nhất


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_row_wrong(arr, k):                    # coi HÀNG (tử cố định) là dãy tăng: sai, hàng giảm dần theo j
    n = len(arr)
    heap = [(arr[i] / arr[i + 1], i, i + 1) for i in range(n - 1)]
    heapq.heapify(heap)
    for _ in range(k - 1):
        _, i, j = heapq.heappop(heap)
        if j + 1 < n:
            heapq.heappush(heap, (arr[i] / arr[j + 1], i, j + 1))
    _, i, j = heap[0]
    return [arr[i], arr[j]]


def bug_heap_wrong_push(arr, k):              # đẩy phần tử kế tiếp theo mẫu (j+1) thay vì tử (i+1)
    n = len(arr)
    heap = [(arr[0] / arr[j], 0, j) for j in range(1, n)]
    heapq.heapify(heap)
    for _ in range(k - 1):
        _, i, j = heapq.heappop(heap)
        if j + 1 < n:
            heapq.heappush(heap, (arr[i] / arr[j + 1], i, j + 1))
    _, i, j = heap[0]
    return [arr[i], arr[j]]


def bug_heap_off_by_one(arr, k):              # pop k lần rồi lấy đỉnh (phân số thứ k+1)
    n = len(arr)
    heap = [(arr[0] / arr[j], 0, j) for j in range(1, n)]
    heapq.heapify(heap)
    for _ in range(k):
        _, i, j = heapq.heappop(heap)
        if i + 1 < j:
            heapq.heappush(heap, (arr[i + 1] / arr[j], i + 1, j))
    _, i, j = heap[0] if heap else (0, 0, 1)
    return [arr[i], arr[j]]


def bug_int_division(arr, k):                 # // thay cho /
    items = sorted((arr[i] // arr[j], arr[i], arr[j]) for i in range(len(arr)) for j in range(i + 1, len(arr)))
    return [items[k - 1][1], items[k - 1][2]]


def bug_stop_ge(arr, k):                      # dừng khi count >= k thay vì count == k
    n = len(arr)
    lo, hi = 0.0, 1.0
    for _ in range(60):
        mid = (lo + hi) / 2
        count, i, bn, bd = 0, 0, 0, 1
        for j in range(1, n):
            while i < j and arr[i] <= mid * arr[j]:
                i += 1
            count += i
            if i > 0 and arr[i - 1] * bd > bn * arr[j]:
                bn, bd = arr[i - 1], arr[j]
        if count >= k:                            # SAI: count có thể > k
            return [bn, bd]
        lo = mid
    return [bn, bd]


BUGS = [bug_row_wrong, bug_heap_wrong_push, bug_heap_off_by_one, bug_int_division]
rng = random.Random(1)
tests = []
for _ in range(400):
    n = rng.randint(2, 7)
    arr = [1] + sorted(rng.sample(PRIMES[:15], n - 1))
    tests.append((arr, rng.randint(1, n * (n - 1) // 2)))
for bug in BUGS:
    found = None
    for arr, k in tests:
        try:
            got = bug(list(arr), k)
        except Exception as e:
            got = type(e).__name__
        if got != kth_brute(arr, k):
            found = (arr, k, got)
            break
    assert found is not None, bug.__name__
    print(f"{bug.__name__:22s} sai trên arr={found[0]}, k={found[1]}: ra {found[2]}, đúng {kth_brute(*found[:2])}")
wrong = sum(1 for arr, k in tests if bug_stop_ge(list(arr), k) != kth_brute(arr, k))
print("bug_stop_ge sai trên", wrong, "/", len(tests), "ca thử")
assert wrong > 0

bug_row_wrong          sai trên arr=[1, 29, 43], k=1: ra [1, 29], đúng [1, 43]
bug_heap_wrong_push    sai trên arr=[1, 3, 19, 41], k=4: ra [1, 3], đúng [3, 19]
bug_heap_off_by_one    sai trên arr=[1, 29, 43], k=1: ra [1, 29], đúng [1, 43]
bug_int_division       sai trên arr=[1, 29, 43], k=1: ra [1, 29], đúng [1, 43]
bug_stop_ge sai trên 231 / 400 ca thử


### D. Vét cạn mọi mảng nhỏ và mọi `k`

Với mọi tập con của 8 số nguyên tố đầu `{2,3,5,7,11,13,17,19}` (kèm số `1` ở đầu) có kích thước `≥ 1` — tổng `255` mảng — và **mọi** `k` hợp lệ, so sánh toàn bộ cài đặt với oracle `Fraction`.

In [21]:
cases = 0
for mask in range(1, 1 << 8):
    arr = [1] + [PRIMES[i] for i in range(8) if mask >> i & 1]
    m = len(arr) * (len(arr) - 1) // 2
    items = sorted((Fraction(arr[i], arr[j]), arr[i], arr[j]) for i in range(len(arr)) for j in range(i + 1, len(arr)))
    for k in range(1, m + 1):
        want = [items[k - 1][1], items[k - 1][2]]
        for name in IMPLS:
            assert IMPLS[name](list(arr), k) == want, (name, arr, k)
        cases += 1
print(cases, "cặp (mảng, k) vét cạn: 9 cài đặt đều khớp oracle")

2816 cặp (mảng, k) vét cạn: 9 cài đặt đều khớp oracle


### E. Ngẫu nhiên có seed, đúng ràng buộc

Mảng ngẫu nhiên: `n` từ 2 đến 60, các số nguyên tố chọn ngẫu nhiên trong `≤ 30000`, `k` ngẫu nhiên hợp lệ (có thiên về các biên).

In [22]:
rng = random.Random(20240786)
count = 0
for trial in range(600):
    n = rng.randint(2, 60)
    arr = [1] + sorted(rng.sample(PRIMES, n - 1))
    m = n * (n - 1) // 2
    ks = {1, m, rng.randint(1, m), rng.randint(1, m), max(1, m // 2)}
    items = sorted((Fraction(arr[i], arr[j]), arr[i], arr[j]) for i in range(n) for j in range(i + 1, n))
    for k in ks:
        want = [items[k - 1][1], items[k - 1][2]]
        for name in IMPLS:
            assert IMPLS[name](list(arr), k) == want, (name, arr, k)
        count += 1
print(count, "cặp (mảng, k) ngẫu nhiên: mọi cài đặt trùng oracle")

# tính chất: đáp án là một cặp (arr[i], arr[j]) với i < j; các phân số liên tiếp theo k tăng chặt
arr = [1] + sorted(rng.sample(PRIMES, 12))
m = len(arr) * (len(arr) - 1) // 2
prev = Fraction(0)
for k in range(1, m + 1):
    a, b = Solution().kthSmallestPrimeFraction(list(arr), k)
    assert a in arr and b in arr and arr.index(a) < arr.index(b)
    cur = Fraction(a, b)
    assert cur > prev
    prev = cur
print("tính chất: k tăng thì phân số tăng chặt, và luôn là cặp (arr[i], arr[j]) với i < j: OK")

2906 cặp (mảng, k) ngẫu nhiên: mọi cài đặt trùng oracle
tính chất: k tăng thì phân số tăng chặt, và luôn là cặp (arr[i], arr[j]) với i < j: OK


### F. Giới hạn lớn nhất

`n = 1000` với 999 số nguyên tố ngẫu nhiên (và cả 999 số nguyên tố đầu tiên, cả 999 số nguyên tố lớn nhất `≤ 3·10^4`); `k` nhỏ, giữa, lớn nhất.

In [23]:
rng = random.Random(777)
sets = {
    "ngẫu nhiên": [1] + sorted(rng.sample(PRIMES, 999)),
    "999 nhỏ nhất": [1] + PRIMES[:999],
    "999 lớn nhất": [1] + PRIMES[-999:],
}
for name, arr in sets.items():
    m = len(arr) * (len(arr) - 1) // 2
    items = sorted((arr[i] / arr[j], arr[i], arr[j]) for i in range(len(arr)) for j in range(i + 1, len(arr)))
    exact_items = None
    for k in (1, 2, 12345, m // 2, m - 1, m):
        want = [items[k - 1][1], items[k - 1][2]]
        for impl in ("heap", "binary", "exact", "Solution", "SolutionBinary", "SolutionExact"):
            if impl in ("heap", "SolutionHeap") and k > 20000:
                continue                                   # heap với k ~ n^2/2 chậm (đúng như phân tích)
            assert IMPLS[impl](list(arr), k) == want, (name, k, impl)
    print(f"{name:14s}: m = {m}, các k đặc biệt: 6 giá trị, mọi cách khớp oracle (sắp xếp số thực)")
# kiểm tra độc lập oracle số thực = oracle Fraction trên một mẫu lớn
arr = sets["ngẫu nhiên"]
sub = rng.sample(range(len(arr)), 3)
print("oracle Fraction và oracle thực khớp trên k = 500:", kth_brute(arr, 500) == [sorted((arr[i] / arr[j], arr[i], arr[j]) for i in range(1000) for j in range(i + 1, 1000))[499][1], sorted((arr[i] / arr[j], arr[i], arr[j]) for i in range(1000) for j in range(i + 1, 1000))[499][2]])

ngẫu nhiên    : m = 499500, các k đặc biệt: 6 giá trị, mọi cách khớp oracle (sắp xếp số thực)


999 nhỏ nhất  : m = 499500, các k đặc biệt: 6 giá trị, mọi cách khớp oracle (sắp xếp số thực)


999 lớn nhất  : m = 499500, các k đặc biệt: 6 giá trị, mọi cách khớp oracle (sắp xếp số thực)


oracle Fraction và oracle thực khớp trên k = 500: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1: nhầm hướng đơn điệu (coi hàng là dãy tăng)

Trong bảng, **cột** (mẫu cố định) tăng khi `i` tăng, nhưng **hàng** (tử cố định) **giảm** khi `j` tăng. Nếu khởi tạo heap bằng phần tử đầu mỗi *hàng* (`arr[i]/arr[i+1]`, phân số *lớn nhất* của hàng) rồi tiến `j + 1`, ta lấy các phân số theo thứ tự không đơn điệu.

In [24]:
arr, k = [1, 2, 3, 5, 7], 4
print("bug_row_wrong:", bug_row_wrong(list(arr), k), "| đúng:", kth_brute(arr, k))
wrong = sum(1 for a, kk in tests if bug_row_wrong(list(a), kk) != kth_brute(a, kk))
print("số ca sai:", wrong, "/", len(tests))
assert wrong > 0

bug_row_wrong: [1, 7] | đúng: [1, 3]
số ca sai: 284 / 400


**Vì sao sai:** heap luôn lấy phần tử nhỏ nhất *trong số các đầu dãy*; điều đó chỉ đúng khi mỗi dãy **tăng dần**. Hàng thì giảm dần nên phần tử kế tiếp của hàng nhỏ hơn đầu dãy, bị "chôn" dưới trong khi lẽ ra phải được lấy trước. **Đúng:** dãy phải là **cột** (tăng theo `i`), hoặc nếu dùng hàng thì phải **bắt đầu từ cuối hàng** (`j = n−1`, phân số nhỏ nhất của hàng) và lùi `j − 1`.

### Lỗi 2: đẩy sai phần tử kế tiếp vào heap

Mỗi cột `j` có tử tăng dần; phần tử kế tiếp của cột là `arr[i+1] / arr[j]` (cùng mẫu). Đẩy `arr[i] / arr[j+1]` (đổi mẫu) làm mất tính đơn điệu "cột tăng dần" và bỏ sót phân số.

In [25]:
arr, k = [1, 2, 3, 5, 7], 6
print("sai :", bug_heap_wrong_push(list(arr), k), "| đúng:", kth_brute(arr, k))
bad = next((a, kk) for a, kk in tests if bug_heap_wrong_push(list(a), kk) != kth_brute(a, kk))
print("ví dụ khác:", bad, "-> sai", bug_heap_wrong_push(list(bad[0]), bad[1]), "đúng", kth_brute(*bad))

sai : [1, 7] | đúng: [3, 7]
ví dụ khác: ([1, 3, 19, 41], 4) -> sai [1, 3] đúng [3, 19]


**Vì sao sai:** heap chỉ chứa phân số nhỏ nhất của mỗi cột; đẩy sai phần tử khiến một số phân số không bao giờ vào heap. **Đúng:** `if i + 1 < j: heappush(..., (arr[i+1] / arr[j], i + 1, j))`.

### Lỗi 3: lệch một lần pop

Đáp án là phân số thứ `k`: sau `k − 1` lần pop thì đỉnh heap là phần tử thứ `k`. Pop `k` lần rồi lấy đỉnh sẽ ra phần tử thứ `k + 1` (và với `k = m` còn làm heap rỗng).

In [26]:
arr, k = [1, 2, 3, 5], 3
print("pop k lần:", bug_heap_off_by_one(list(arr), k), "| đúng:", kth_brute(arr, k), "(phần tử thứ k+1 là 1/2)")
assert bug_heap_off_by_one(list(arr), k) == [1, 2]

pop k lần: [1, 2] | đúng: [2, 5] (phần tử thứ k+1 là 1/2)


**Vì sao sai:** nhầm "bỏ `k − 1` phần tử rồi đọc" với "bỏ `k` phần tử". **Đúng:** `for _ in range(k - 1)`.

### Lỗi 4: dùng phép chia nguyên `//`

`arr[i] // arr[j]` luôn bằng `0` (vì tử `<` mẫu) nên mọi phân số "bằng nhau" và việc sắp xếp mất ý nghĩa.

In [27]:
arr, k = [1, 2, 3, 5], 3
print("//:", bug_int_division(list(arr), k), "| đúng:", kth_brute(arr, k))
print("giá trị khoá của mọi phân số:", {arr[i] // arr[j] for i in range(4) for j in range(i + 1, 4)})

//: [1, 5] | đúng: [2, 5]
giá trị khoá của mọi phân số: {0}


**Vì sao sai:** khoá luôn `0` → sắp theo `(0, tử, mẫu)`, tức theo tử rồi mẫu, không phải theo giá trị. **Đúng:** `/` (hoặc `Fraction`, hoặc nhân chéo).

### Lỗi 5: dừng khi `count >= k` thay vì `count == k`

Nếu `count(mid) > k` thì có **nhiều hơn `k`** phân số `≤ mid`, phân số lớn nhất trong số đó là phân số thứ `count`, không phải thứ `k`. Phải thu hẹp khoảng (`hi = mid`) cho tới khi đếm được **đúng** `k`.

In [28]:
wrong = [(a, kk) for a, kk in tests if bug_stop_ge(list(a), kk) != kth_brute(a, kk)]
print("số ca sai của bản 'count >= k':", len(wrong), "/", len(tests))
a, kk = wrong[0]
print("ví dụ:", a, kk, "-> bản sai:", bug_stop_ge(list(a), kk), "| đúng:", kth_brute(a, kk))

số ca sai của bản 'count >= k': 231 / 400
ví dụ: [1, 29, 43] 1 -> bản sai: [1, 29] | đúng: [1, 43]


**Vì sao sai:** hàm `count` nhảy từng bước theo các phân số; `count >= k` có thể dừng ở `mid` đã vượt phân số thứ `k`. **Đúng:** `if count == k: return ...; if count < k: lo = mid else: hi = mid`. (Cũng nhớ dùng `<=` khi đếm: `count(x)` là số phân số **không vượt quá** `x`.)

### Lỗi 6 (bonus): trả về giá trị thực thay vì cặp số

Đề yêu cầu `[arr[i], arr[j]]`, không phải `arr[i] / arr[j]`. Luôn ghi nhận **chỉ số / giá trị tử và mẫu** cùng lúc với giá trị phân số (trong heap: `(giá trị, i, j)`; trong nhị phân: `best_num, best_den`).

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tìm phần tử nhỏ thứ `k` trong `O(n²)` ứng viên có cấu trúc đơn điệu theo hàng/cột" → hai hướng chuẩn: **min-heap gộp các dãy đã sắp** (`O(k log n)`) hoặc **nhị phân theo giá trị + đếm bằng hai con trỏ** (`O(n log C)`).

**Mẹo:**

1. Nhìn bảng `i × j`: hàng/cột nào đơn điệu? Đó là chìa khoá.
2. Muốn tránh sai số số thực: so sánh bằng **nhân chéo số nguyên**, hoặc `Fraction`.
3. Với nhị phân theo giá trị: định nghĩa `count(x)` rõ ràng (≤ hay <), điều kiện dừng `count == k` hoặc kiểu "tìm biên".
4. Khi nhị phân thực: đảm bảo khoảng cách giữa hai ứng viên khác nhau lớn hơn sai số làm tròn.
5. Luôn có **oracle vét cạn** (sắp xếp mọi cặp bằng `Fraction`) để kiểm chứng.

**Họ bài cùng pattern:** Kth Smallest Element in a Sorted Matrix (378), Find K Pairs with Smallest Sums (373), Find K-th Smallest Pair Distance (719), Kth Smallest Number in Multiplication Table (668), Ugly Number II (264).

## 15. Biến thể

1. **Phân số lớn thứ `k`**: đối xứng, lấy `k' = m − k + 1` nhỏ thứ `k'`, hoặc dùng max-heap.
2. **Trả về mọi `k` phân số nhỏ nhất** (đúng thứ tự): chạy heap, ghi từng lần pop.
3. **Mảng có thể trùng / không nguyên tố**: các phân số có thể bằng nhau; cần định nghĩa lại "thứ `k`" (có tính trùng không?) và cách xử lý hòa.
4. **Ghép phân số khác**: `arr[j] / arr[i]` với `i < j` (các phân số `> 1`).
5. **Số phân số `≤ x`** (hàm đếm): chính là `count(x)` của cách nhị phân, dùng riêng.

Cài đặt biến thể 2 và 5:

In [29]:
def smallest_k_fractions(arr, k):
    """Biến thể 2: k phân số nhỏ nhất theo thứ tự (heap)."""
    n = len(arr)
    heap = [(arr[0] / arr[j], 0, j) for j in range(1, n)]
    heapq.heapify(heap)
    out = []
    for _ in range(k):
        _, i, j = heapq.heappop(heap)
        out.append((arr[i], arr[j]))
        if i + 1 < j:
            heapq.heappush(heap, (arr[i + 1] / arr[j], i + 1, j))
    return out


def count_le(arr, p, q):
    """Biến thể 5: số cặp (i < j) có arr[i]/arr[j] <= p/q (nhân chéo, chính xác)."""
    count, i = 0, 0
    for j in range(1, len(arr)):
        while i < j and arr[i] * q <= p * arr[j]:
            i += 1
        count += i
    return count


arr = [1, 2, 3, 5, 7]
print(smallest_k_fractions(arr, 5))
assert smallest_k_fractions(arr, 5) == [(a, b) for _, a, b in sorted((Fraction(arr[i], arr[j]), arr[i], arr[j]) for i in range(5) for j in range(i + 1, 5))][:5]
print("số phân số <= 1/2:", count_le(arr, 1, 2), "| <= 1:", count_le(arr, 1, 1), "(tổng", 5 * 4 // 2, ")")
assert count_le(arr, 1, 2) == sum(1 for i in range(5) for j in range(i + 1, 5) if Fraction(arr[i], arr[j]) <= Fraction(1, 2))

[(1, 7), (1, 5), (2, 7), (1, 3), (2, 5)]
số phân số <= 1/2: 7 | <= 1: 10 (tổng 10 )


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê + sắp xếp | Sinh `n²/2` phân số, sắp, lấy thứ `k` | `O(n² log n)` | `O(n²)` | Làm oracle / `n` rất nhỏ |
| 2. Min-heap gộp cột | Mỗi mẫu `arr[j]` là một dãy tăng; gộp `k` lần | `O(n + k log n)` | `O(n)` | `k` nhỏ; ý tưởng tổng quát cho "k-th trong ma trận sắp xếp" |
| 3. Nhị phân + hai con trỏ | Nhị phân trên giá trị, đếm `≤ x` bằng hai con trỏ | `O(n log(1/δ))` | `O(1)` | **Đáp án chính**; đáp ứng Follow up |
| 4. Nhị phân `Fraction` | Như Cách 3 nhưng số học chính xác | `O(n log(1/δ))` | `O(1)` | Muốn bảo đảm tuyệt đối không dựa vào số thực |

Cả bốn cho cùng đáp án trên hơn `2300` cặp (mảng, `k`) vét cạn và `3000` cặp ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [30]:
arr = [1, 2, 3, 7]
print(sorted(f"{arr[i]}/{arr[j]}" for i in range(4) for j in range(i + 1, 4)))
print(len(arr) * (len(arr) - 1) // 2)
print(kth_brute(arr, 1), kth_brute(arr, 6), kth_brute(arr, 3))

['1/2', '1/3', '1/7', '2/3', '2/7', '3/7']
6
[1, 7] [2, 3] [1, 3]


### Bài 2 (tính tay)

`arr = [1, 2, 3, 5, 7]`.

- (a) Có bao nhiêu phân số? Liệt kê chúng theo thứ tự tăng dần (cần so sánh bằng nhân chéo).
- (b) Phân số nhỏ thứ 4 và nhỏ thứ 8 là gì?
- (c) Có bao nhiêu phân số `≤ 1/2`? (đây chính là `count(0,5)` của nhị phân).

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cách heap:

In [31]:
def kth_heap_todo(arr, k):
    n = len(arr)
    heap = [(arr[0] / arr[j], 0, j) for j in range(1, n)]
    heapq.heapify(heap)
    for _ in range(k - 1):
        _, i, j = heapq.heappop(heap)
        # TODO: nếu còn tử kế tiếp trong cùng cột j (i + 1 < j) thì đẩy (arr[i+1]/arr[j], i+1, j) vào heap
        pass
    # TODO: lấy đỉnh heap và trả về [arr[i], arr[j]]
    return None


# Khi làm xong, bỏ comment để kiểm tra:
# assert kth_heap_todo([1, 2, 3, 5], 3) == [2, 5]
# assert kth_heap_todo([1, 7], 1) == [1, 7]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết lớp `MySolution` bằng heap hoặc nhị phân:

In [32]:
class MySolution:
    def kthSmallestPrimeFraction(self, arr, k):
        pass            # TODO


# Khi làm xong:
# for a, kk, w in [([1, 2, 3, 5], 3, [2, 5]), ([1, 7], 1, [1, 7]), ([1, 2, 3, 5, 7], 4, [2, 7])]:
#     assert MySolution().kthSmallestPrimeFraction(list(a), kk) == w

### Bài 5 (mini challenge): đếm số phân số trong khoảng `(L, R]`

Viết `count_between(arr, L, R)` với `L = (p1, q1)`, `R = (p2, q2)` (phân số `p1/q1 < p2/q2`) trả về số cặp `i < j` thoả `p1/q1 < arr[i]/arr[j] <= p2/q2`, trong `O(n)` (hai con trỏ), dùng **nhân chéo** (không số thực). Gợi ý: `count_between = count_le(R) − count_le(L)`.

In [33]:
def count_between(arr, L, R):
    pass            # TODO


# Khi làm xong:
# assert count_between([1, 2, 3, 5, 7], (1, 5), (1, 2)) == 5

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: số phân số là `n(n−1)/2`.
- Mức 2: `[1,2,3,7]` có `6` phân số: `1/2, 1/3, 1/7, 2/3, 2/7, 3/7`.
- Mức 3: sắp: `1/7 < 2/7 < 1/3 < 3/7 < 1/2 < 2/3`. Nhỏ thứ 1 là `1/7`, thứ 6 là `2/3`, thứ 3 là `1/3`.

**Bài 2**

- Mức 1: có `10` phân số.
- Mức 2: so sánh từng cặp bằng nhân chéo (ví dụ `2/7` so `1/3`: `2·3 = 6 < 1·7 = 7` nên `2/7 < 1/3`); các phân số nhỏ nhất đều có mẫu lớn.
- Mức 3: chạy `sorted` với `Fraction` để xác minh.

**Bài 3**

- Mức 1: giống cell trace ở mục 8.
- Mức 2: sau pop, nếu `i + 1 < j` thì `heappush(heap, (arr[i+1]/arr[j], i+1, j))`.
- Mức 3: cuối: `_, i, j = heap[0]; return [arr[i], arr[j]]`.

**Bài 4**

- Mức 1: chọn heap (đơn giản) hoặc nhị phân (nhanh).
- Mức 2: nhớ chỉ lấy cặp `i < j`.
- Mức 3: xem mục 9 (nhị phân) hoặc Cách 2 (heap).

**Bài 5**

- Mức 1: `count_le(p, q)` đã có ở mục 15.
- Mức 2: số phân số trong `(L, R]` bằng số `≤ R` trừ số `≤ L`.
- Mức 3: `count_le(arr, *R) - count_le(arr, *L)`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [34]:
arr = [1, 2, 3, 7]
assert sorted(f"{arr[i]}/{arr[j]}" for i in range(4) for j in range(i + 1, 4)) == sorted(["1/2", "1/3", "1/7", "2/3", "2/7", "3/7"])
assert (kth_brute(arr, 1), kth_brute(arr, 6), kth_brute(arr, 3)) == ([1, 7], [2, 3], [1, 3])
print("Bài 1 OK")

Bài 1 OK


### Bài 2

(a) Có `5·4/2 = 10` phân số; ta để máy sắp tăng bằng `Fraction` để tránh sai sót (xem kết quả in ra bên dưới). (b) Thứ 4 và thứ 8 đọc từ danh sách đã sắp. (c) Số phân số `≤ 1/2` đếm trực tiếp, và phải khớp hàm hai con trỏ `count_le`.

In [35]:
arr = [1, 2, 3, 5, 7]
order = sorted((Fraction(arr[i], arr[j]), arr[i], arr[j]) for i in range(5) for j in range(i + 1, 5))
print([f"{a}/{b}" for _, a, b in order])
assert len(order) == 10
print("thứ 4:", order[3][1:], "| thứ 8:", order[7][1:])
n_half = sum(1 for f, _, _ in order if f <= Fraction(1, 2))
assert count_le(arr, 1, 2) == n_half
print("số phân số <= 1/2:", n_half)
for k in range(1, 11):
    assert Solution().kthSmallestPrimeFraction(list(arr), k) == [order[k - 1][1], order[k - 1][2]]
print("Bài 2 OK")

['1/7', '1/5', '2/7', '1/3', '2/5', '3/7', '1/2', '3/5', '2/3', '5/7']
thứ 4: (1, 3) | thứ 8: (3, 5)
số phân số <= 1/2: 7
Bài 2 OK


### Bài 3

In [36]:
def kth_heap_answer(arr, k):
    n = len(arr)
    heap = [(arr[0] / arr[j], 0, j) for j in range(1, n)]
    heapq.heapify(heap)
    for _ in range(k - 1):
        _, i, j = heapq.heappop(heap)
        if i + 1 < j:
            heapq.heappush(heap, (arr[i + 1] / arr[j], i + 1, j))
    _, i, j = heap[0]
    return [arr[i], arr[j]]


assert kth_heap_answer([1, 2, 3, 5], 3) == [2, 5] and kth_heap_answer([1, 7], 1) == [1, 7]
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def kthSmallestPrimeFraction(self, arr, k):
        n = len(arr)
        heap = [(arr[0] / arr[j], 0, j) for j in range(1, n)]
        heapq.heapify(heap)
        for _ in range(k - 1):
            _, i, j = heapq.heappop(heap)
            if i + 1 < j:
                heapq.heappush(heap, (arr[i + 1] / arr[j], i + 1, j))
        _, i, j = heap[0]
        return [arr[i], arr[j]]


for a, kk, w in [([1, 2, 3, 5], 3, [2, 5]), ([1, 7], 1, [1, 7]), ([1, 2, 3, 5, 7], 4, kth_brute([1, 2, 3, 5, 7], 4))]:
    assert MySolutionAnswer().kthSmallestPrimeFraction(list(a), kk) == w
rng = random.Random(4)
for _ in range(200):
    n = rng.randint(2, 12)
    a = [1] + sorted(rng.sample(PRIMES, n - 1))
    kk = rng.randint(1, n * (n - 1) // 2)
    assert MySolutionAnswer().kthSmallestPrimeFraction(list(a), kk) == kth_brute(a, kk)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

Số phân số trong khoảng nửa mở `(L, R]` bằng số phân số `≤ R` trừ số phân số `≤ L`.

In [38]:
def count_between_answer(arr, L, R):
    return count_le(arr, *R) - count_le(arr, *L)


assert count_between_answer([1, 2, 3, 5, 7], (1, 5), (1, 2)) == sum(
    1 for i in range(5) for j in range(i + 1, 5) if Fraction(1, 5) < Fraction([1, 2, 3, 5, 7][i], [1, 2, 3, 5, 7][j]) <= Fraction(1, 2))
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(2, 10)
    a = [1] + sorted(rng.sample(PRIMES, n - 1))
    q1, q2 = rng.randint(1, 30), rng.randint(1, 30)
    p1, p2 = rng.randint(0, q1), rng.randint(0, q2)
    L, R = (p1, q1), (p2, q2)
    if Fraction(*L) >= Fraction(*R):
        continue
    want = sum(1 for i in range(n) for j in range(i + 1, n) if Fraction(*L) < Fraction(a[i], a[j]) <= Fraction(*R))
    assert count_between_answer(a, L, R) == want
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (phần tử nhỏ thứ `k` trong các dãy đã sắp, kiểu heap):**

```python
class Solution:
    def kthSmallestPrimeFraction(self, arr, k):
        n = len(arr)
        heap = [(arr[0] / arr[j], 0, j) for j in range(1, n)]   # đầu mỗi cột
        heapq.heapify(heap)
        for _ in range(k - 1):
            _, i, j = heapq.heappop(heap)
            if i + 1 < j:
                heapq.heappush(heap, (arr[i + 1] / arr[j], i + 1, j))
        _, i, j = heap[0]
        return [arr[i], arr[j]]
```

**Template (nhị phân trên giá trị):**

```python
lo, hi = 0.0, 1.0
while True:
    mid = (lo + hi) / 2
    count, i, (bn, bd) = 0, 0, (0, 1)
    for j in range(1, n):
        while i < j and arr[i] <= mid * arr[j]:
            i += 1
        count += i
        if i and arr[i - 1] * bd > bn * arr[j]:
            bn, bd = arr[i - 1], arr[j]
    if count == k: return [bn, bd]
    lo, hi = (mid, hi) if count < k else (lo, mid)
```

**Checklist trước khi submit:**

- [ ] Chỉ xét `i < j`?
- [ ] Heap: đẩy `arr[i+1]/arr[j]` (cùng mẫu), điều kiện `i + 1 < j`, pop đúng `k − 1` lần?
- [ ] Nhị phân: đếm bằng `<=`, hai con trỏ chỉ tiến, ghi nhận phân số lớn nhất `≤ mid` bằng nhân chéo?
- [ ] Dùng `/` (không phải `//`); trả về **cặp số** `[tử, mẫu]`, không phải giá trị thực?
- [ ] Đã thử `k = 1` (`1/arr[-1]`), `k = m` (tỉ số liền kề lớn nhất), `n = 2`?
- [ ] Đúng tên `class Solution`, `kthSmallestPrimeFraction(self, arr, k)`, không in debug.